<h1 style="text-align:center; color: #2E86C1; font-family:Arial;"> Practice Multi Agentic AI with CrewAI </h1>

In [1]:
from crewai import Crew, Task, Agent, Process, LLM
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
from langchain_community.tools import BraveSearch
from typing import Type, Optional
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["BRAVE_SEARCH_API_KEY"] = userdata.get("BRAVE_SEARCH")

In [2]:
from crewai.tools import tool

In [7]:
# Create the .ssh directory if it doesn't exist
!mkdir -p ~/.ssh

# Copy the private and public keys from Google Drive
!cp '/content/drive/MyDrive/Colab_SSH_Keys/id_rsa' ~/.ssh/id_rsa
!cp '/content/drive/MyDrive/Colab_SSH_Keys/id_rsa.pub' ~/.ssh/id_rsa.pub

# Set strict permissions (crucial for SSH)
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_rsa
!chmod 644 ~/.ssh/id_rsa.pub

# Add GitHub to known_hosts to prevent host key verification issues
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

# github.com:22 SSH-2.0-23f59ca
# github.com:22 SSH-2.0-23f59ca
# github.com:22 SSH-2.0-23f59ca
# github.com:22 SSH-2.0-23f59ca
# github.com:22 SSH-2.0-23f59ca


In [8]:
!git clone -b practice-genai-agentic-ai git@github.com:YashwanthMRamachandra/GenAI-and-AgenticAI-Practice.git

Cloning into 'GenAI-and-AgenticAI-Practice'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 88 (delta 25), reused 69 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 57.98 KiB | 751.00 KiB/s, done.
Resolving deltas: 100% (25/25), done.


In [3]:
llm = LLM(
    model="gemini-2.5-flash",
    api_key=os.environ["GEMINI_API_KEY"]
)

## Define BraveSearch Input Schema

In [4]:
class BraveSearchInput(BaseModel):
    """Input schema for BraveSearch Tool."""
    query: str = Field(..., description="The search query to execute")
    count: Optional[int] = Field(default=3,
                                 description="Number of search results to return (max 20)")

@tool("SearchTool")
def brave_search_wrapper(query: str) -> str:
    """Search the web for information on a given topic using Brave Search."""
    brave_search = BraveSearch.from_api_key(
        api_key=os.environ["BRAVE_SEARCH_API_KEY"],
        search_kwargs={"count": 3}
    )

    result = brave_search.run(query)
    return result

# def brave_search_wrapper(*args, **kwargs):
#     if isinstance(kwargs, dict) and "query" in kwargs:
#         query = kwargs["query"]
#     elif len(args) > 0 and isinstance(args[0], BraveSearchInput):
#         query = args[0].query
#     else:
#         raise ValueError("Invalid input provided to BraveSearchTool.")

#     brave_search = BraveSearch.from_api_key(
#         api_key=os.environ["BRAVE_SEARCH_API_KEY"],
#         search_kwargs={"count": 3}
#     )

#     result = brave_search.run(query)
#     return result

In [5]:
# def create_brave_search_tool():
#     return CrewStructuredTool.from_function(
#         name="brave_search_tool",
#         description=(
#             "Searches the web using BraveSearch and returns relevant information for a given query. "
#             "Useful for finding up-to-date and accurate information on a wide range of topics."
#         ),
#         args_schema=BraveSearchInput,  # Use the BraveSearch input schema
#         func=brave_search_wrapper
#     )

In [6]:
# Create BraveSearch Tool
# BraveSearchTool = create_brave_search_tool()

## Web Searcher Agent

In [7]:
web_researcher_agent = Agent(
    role="Web Research Specialist",
    goal=(
        "To find the most recent, impactful, and relevant about {topic}. This includes identifying "
        "key use cases, challenges, and statistics to provide a foundation for deeper analysis."
    ),
    backstory=(
        "You are a former investigative journalist known for your ability to uncover technology breakthroughs "
        "and market insights. With years of experience, you excel at identifying actionable data and trends."
    ),
    tools=[brave_search_wrapper], # Fixed: instantiate the tool class
    llm=llm,
    verbose=True
)

## Trend Analyst Agent

In [8]:
trend_analyst_agent = Agent(
    role="Insight Synthesizer",
    goal=(
        "To analyze research findings, extract significant trends, and rank them by industry impact, growth potential, "
        "and uniqueness. Provide actionable insights for decision-makers."
    ),
    backstory=(
        "You are a seasonal strategy consultant who transitioned into {topic} analysis. With an eye for patterns, "
        "you specialize in translating raw data into clear, actionable insights."
    ),
    tools=[],
    llm=llm,
    verbose=True
)

## Report Writer Agent

In [9]:
report_writer_agent = Agent(
    role="Narrative Architect",
    goal=(
        "To craft a detailed, professional report that communicates research findings and analysis effectively. "
        "Focus on clarity, logical flow and engagement."
    ),
    backstory=(
        "Once a technical writer for a renowned journal, you are now dedicated to creating industry-leading reports. "
        "You blend storytelling with data to ensure tour work is both informative and captivating."
    ),
    tools=[],
    llm=llm,
    verbose=True
)

## Proof Reader Agent

In [10]:
proof_reader_agent = Agent(
    role="Polisher of Excellence",
    goal=(
      "To refine the report for grammatical accuracy, readability and formatting, ensuring it meets professional "
      "publication standards."
    ),
    backstory=(
        "An award-winning editor turned proofreader, you specialize in perfecting written content. Your sharpe eye for "
        "details ensures every document is flawless."
    ),
    tools=[],
    llm=llm,
    verbose=True
)

## Manager Agent

In [11]:
manager_agent = Agent(
    role="Workflow Maestro",
    goal=(
        "To coordinate agents, manage task dependencies and ensure all outputs meets quality standards. Your focus "
        "is on delivering a cohesive final product through efficient task management."
    ),
    backstory=(
        "A former project manager with a passion for efficient teamwork, you ensure every process run smoothly, "
        "overseeing tasks and verifying results."
    ),
    tools=[],
    llm=llm,
    verbose=True
)

## Create Tasks

In [12]:
web_research_task = Task(
    description=(
        "Conduct web-based research to identify 5-7 of the {topic}. Focus on key use cases."
    ),
    expected_output=(
        "A structured list of 5-7 {topic}"
    )
)

In [13]:
trend_analysis_task = Task(
    description=(
        "Analyze the research findings to rank {topic}."
    ),
    expected_output=(
        "A table ranking trends by impact, with concise descriptions of each trend."
    )
)

In [14]:
report_writing_task = Task(
    description=(
        "Draft report summarizing the findings and analysis of {topic}. Include sections for "
        "Introduction, Trends Overview, Analysis, and Recommendations."
    ),
    expected_output=(
        "A structured, professional draft with a clear flow of information. Ensure logical organization and consistent tone."
    )
)

In [15]:
proof_reading_task = Task(
    description=(
        "Refine the task for grammatical accuracy, coherence and formatting. Ensure the final document is polished "
        "and ready for publication."
    ),
    expected_output=(
        "A professional, polished report free of grammatical errors and inconsistencies. Format the document for "
        "easy readability."
    )
)

## Create Crew(Orchestrator)

In [16]:
from langchain_core.callbacks import manager
crew = Crew(
    agents=[web_researcher_agent, trend_analyst_agent, report_writer_agent, proof_reader_agent],
    tasks=[web_research_task, trend_analysis_task, report_writing_task, proof_reading_task],
    process=Process.hierarchical,
    manager_agent=manager_agent,
    verbose=True
)

In [ ]:
crew_output = crew.kickoff(inputs={"topic": "Generative AI in Healthcare"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d5a4ca15-cb72-440e-93fe-f8e7095203f6                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct web-based research to identify 5-7 of the Generative AI in Healthcare. Focus on key use cases.   │
│  ID: 753d2723-e4fa-45b8-b7ac-94bb04d10284                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Maestro                                                                                        │
│                                                                                                                 │
│  Task: Conduct web-based research to identify 5-7 of the Generative AI in Healthcare. Focus on key use cases.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'The final output should be a structured list of 5-7 Generative AI in Healthcare use cases.  │
│  You MUST return the actual complete content as the final answer, not a summary.', 'task': 'Ident...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Research Specialist                                                                                 │
│                                                                                                                 │
│  Task: Identify 5-7 key use cases of Generative AI in Healthcare.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Maestro                                                                                        │
│                                                                                                                 │
│  Task: Analyze the research findings to rank Generative AI in Healthcare.                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Maestro                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Rank | Trend/Use Case                           | Description                                                │
│  |                                                                                                              │
│  |------|------------------------------------------|----------------------------------------------------------  │
│  -------------------------------------------------------------------------------------------------------------  │
│  ------------------------------------------------------------------------|                                      │
│  | 1    | Personalized Medicine and Treatment Planning | AI models analyze patient data (genomic, EHR,          │
│  imaging) to generate tailored treatment plans, predict individual therapy responses, and recommend optimal     │
│  interventions for more effective, personalized care.                                     |                     │
│  | 2    | Drug Discovery and Development           | Generative AI designs novel molecular structures,          │
│  accelerates the identification of drug candidates, optimizes existing drugs, and predicts drug-target          │
│  interactions, streamlining the drug development pipeline.                                |                     │
│  | 3    | Predictive Diagnostics and Early Disease Detection | Generative AI learns from complex health         │
│  datasets to identify subtle patterns and generate early predictions of disease onset or progression, enabling  │
│  proactive interventions and improving patient outcomes through timely diagnosis. |                             │
│  | 4    | Medical Image Synthesis and Augmentation | Generative AI creates realistic synthetic medical images   │
│  (e.g., X-rays, MRIs) for training diagnostic AI models, augmenting limited datasets, and simulating various    │
│  disease states, enhancing robust diagnostic tools.                               |                             │
│  | 5    | Synthetic Data Generation for Research   | To address data privacy and scarcity, generative models    │
│  produce high-fidelity synthetic patient data replicating real data's statistical characteristics, enabling     │
│  extensive research and algorithm development.                               |                                  │
│  | 6    | Automated Report Generation and Documentation | Generative AI assists healthcare professionals by     │
│  automatically generating clinical notes, patient summaries, discharge instructions, and other administrative   │
│  documents, reducing clerical tasks and improving documentation efficiency.   |                                 │
│  | 7    | Virtual Assistants and Chatbots for Patient Engagement | AI-powered conversational agents provide     │
│  natural language responses to patient queries, offer health information, assist with appointment scheduling,   │
│  and deliver support, improving patient access, education, and engagement.          |                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Analyze the research findings to rank Generative AI in Healthcare.                                             │
│  Agent:                                                                                                         │
│  Workflow Maestro                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Draft report summarizing the findings and analysis of Generative AI in Healthcare. Include sections for  │
│  Introduction, Trends Overview, Analysis, and Recommendations.                                                  │
│  ID: 1cac938f-415e-4133-b0c7-c1e04e139066                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Maestro                                                                                        │
│                                                                                                                 │
│  Task: Draft report summarizing the findings and analysis of Generative AI in Healthcare. Include sections for  │
│  Introduction, Trends Overview, Analysis, and Recommendations.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [97]:
!cp '/content/drive/MyDrive/Colab Notebooks/Agentic AI - Multi Agent with CrewAI.ipynb' \
    '/content/GenAI-and-AgenticAI-Practice'